#### 1. Data formatting to tensors

In [1]:
import os
import sys
from pathlib import Path

# find the main project root folder automatically
current_dir = Path().resolve()
project_root = current_dir
while project_root != project_root.parent:
    if (project_root / "src").exists():
        break
    project_root = project_root.parent

# add src to sys.path so your imports still work
src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.append(src_path)

# set the working directory to the 'src' folder
os.chdir(project_root / "src")

In [2]:
#0. Import libaries 
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn import metrics
from protgen.data import ProteinDataset
from torch.utils.data import DataLoader
from protgen.data.make_data import IGNORE_INDEX

/opt/anaconda3/lib/python3.12/site-packages/matplotlib/tri/_triinterpolate.py:665: RuntimeWarning: divide by zero encountered in divide
  gauss_w = np.ones([9], dtype=np.float64) / 9.


In [3]:
#1. Data 
train_data = ProteinDataset(path_to_data="../data/train.csv")
val_data = ProteinDataset(path_to_data="../data/val.csv")
test_data = ProteinDataset(path_to_data="../data/test.csv")

print(f"Data class: {train_data}")

# Get tensors
X_train = train_data.tokens
y_train_shifted = train_data.tokens_shifted # generative task

print(f"# Train data: \n Tokens shape: {X_train.shape}\n Shifted shape: {y_train_shifted.shape}")

# Inddexing
x, x_shifted, y, length = train_data[0]
print(f"# Sample 0: \nTokens: \n {x}\n Shifted: \n {x_shifted}\n Label: {y}\n Length: {length}")


Data class: ProteinDataset(num_samples=16020, vocab_size=23, num_classes=3, max_length=129)
# Train data: 
 Tokens shape: torch.Size([16020, 129])
 Shifted shape: torch.Size([16020, 129])
# Sample 0: 
Tokens: 
 tensor([ 1, 13, 10, 11, 11, 16,  7, 14, 10, 10, 11, 14, 19,  8,  7,  3, 17, 15,
         3, 19, 12, 12, 20, 11, 12, 18, 16, 17,  7, 19, 18,  5, 10, 18, 12,  6,
        22, 16,  8, 11, 19, 10, 14, 12, 11,  9, 18,  3, 11, 18, 10, 13,  5, 10,
        13, 18, 12, 18, 10, 17, 15,  8, 19, 11,  7, 22, 10, 11,  3,  6,  8, 10,
         5,  6, 20, 16,  3, 10, 16, 12, 10, 16,  5, 22, 12, 22, 11,  6, 11, 12,
        10, 14, 22,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0])
 Shifted: 
 tensor([13, 10, 11, 11, 16,  7, 14, 10, 10, 11, 14, 19,  8,  7,  3, 17, 15,  3,
        19, 12, 12, 20, 11, 12, 18, 16, 17,  7, 19, 18,  5, 10, 18, 12,  6, 22,
        16,  8, 11, 19, 10, 14, 12, 

In [4]:
import torch
import torch.nn.functional as F
from protgen.data import AA, FAMILIES, IGNORE_INDEX, special_tokens

print(f"vocab ({train_data.vocab_size} tokens): {train_data.vocab}")
print(f"families: {train_data.fam_to_idx}")
print(f"padding is ignored in the loss with ignore_index={IGNORE_INDEX}")

# DataLoader
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

sequence = train_data.df["sequence"][0]
tokens = train_data.encode(sequence)
print(sequence)
print(tokens)
print()
print(f"encode / decode round-trips: {train_data.decode(tokens) == sequence}")

x, x_shifted, y, length = next(iter(train_loader))
print(f"batch: x {tuple(x.shape)} {x.dtype}, x_shifted {tuple(x_shifted.shape)}, y {tuple(y.shape)}")

# Generative: one distribution over the vocabulary per position.
logits_gen = torch.randn(x.shape[0], train_data.vocab_size, train_data.max_length)
print(f"generative loss:     {F.cross_entropy(logits_gen, x_shifted, ignore_index=IGNORE_INDEX):.4f}")

vocab (23 tokens): {'<PAD>': 0, '<BOS>': 1, '<EOS>': 2, 'A': 3, 'C': 4, 'D': 5, 'E': 6, 'F': 7, 'G': 8, 'H': 9, 'I': 10, 'K': 11, 'L': 12, 'M': 13, 'N': 14, 'P': 15, 'Q': 16, 'R': 17, 'S': 18, 'T': 19, 'V': 20, 'W': 21, 'Y': 22}
families: {'AcyP': 0, 'HPr': 1, 'HU': 2}
padding is ignored in the loss with ignore_index=-1
MIKKQFNIIKNTGFARPATLLVKLSQRFTSDISLEYQGKTINLKHSAKSIMDIMSLSIRPGTKFYIKAEGIDEVQAIQLIQDYLYKEKLINY
tensor([ 1, 13, 10, 11, 11, 16,  7, 14, 10, 10, 11, 14, 19,  8,  7,  3, 17, 15,
         3, 19, 12, 12, 20, 11, 12, 18, 16, 17,  7, 19, 18,  5, 10, 18, 12,  6,
        22, 16,  8, 11, 19, 10, 14, 12, 11,  9, 18,  3, 11, 18, 10, 13,  5, 10,
        13, 18, 12, 18, 10, 17, 15,  8, 19, 11,  7, 22, 10, 11,  3,  6,  8, 10,
         5,  6, 20, 16,  3, 10, 16, 12, 10, 16,  5, 22, 12, 22, 11,  6, 11, 12,
        10, 14, 22,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0])

enc

In [5]:
print(x)
print(x.shape)
print(x.dtype)
print(train_data[3])

tensor([[ 1, 13, 19,  ...,  0,  0,  0],
        [ 1, 13, 19,  ...,  0,  0,  0],
        [ 1, 13, 15,  ...,  0,  0,  0],
        ...,
        [ 1, 13, 14,  ...,  0,  0,  0],
        [ 1, 13, 19,  ...,  0,  0,  0],
        [ 1, 13, 15,  ...,  0,  0,  0]])
torch.Size([32, 129])
torch.int64
(tensor([ 1, 13, 14, 11, 19,  6, 12, 10, 12, 17, 10,  3,  6,  6, 19,  8,  7, 18,
        17, 11, 14, 20,  6, 12,  8, 12, 14,  3,  3, 10, 14, 12, 10, 19, 11,  3,
        12, 19,  6,  6,  6, 11, 20, 16, 10, 20,  8,  7,  8,  3,  7,  6, 19, 11,
        20, 17,  3,  6, 17, 13,  8, 17, 14, 15, 11, 19,  8,  6,  5, 10, 15, 10,
        15,  3, 19, 11, 20, 15, 20,  7, 11,  3,  8, 11,  3, 12, 11,  5,  3, 20,
         5, 16,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0]), tensor([13, 14, 11, 19,  6, 12, 10, 12, 17, 10,  3,  6,  6, 19,  8,  7, 18, 17,
        11, 14, 20,  6, 12,  8, 12, 14,  3,  3, 1

#### 2. Model (not working yet, will keep working on this)

In [6]:
class MLP(nn.Module):
    "Initial multilayer perceptron"
    def __init__(self, hidden_layers, hidden_layers2, embedding_dim, context_window=3):
        super(MLP, self).__init__()

        voc = train_data.vocab_size
        activation_fn = nn.ReLU
        self.context_window = context_window

        self.flatten = nn.Flatten()
        input_dim = self.context_window * embedding_dim
        
        # Embedding
        self.embedding = nn.Embedding(num_embeddings=voc, embedding_dim=embedding_dim, padding_idx=IGNORE_INDEX)

        # Slightly expanded hidden layers to give the model more learning capacity
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_layers),
            activation_fn(),
            nn.Linear(hidden_layers, hidden_layers2),
            nn.Linear(hidden_layers2, train_data.vocab_size)
        )

    def forward(self, x):
        step1 = self.embedding(x)
        step1 = self.flatten(step1)
        step2 = self.net(step1)
        return step2

#### 3. Train (heavily inspired by 4.2 CNN CIFAR-10)

#3.1 Define loss and optimizer

In [7]:
# Initialization
model = MLP(hidden_layers=64, hidden_layers2=32, embedding_dim=32, context_window=3)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Verify no NaNs exist
has_nan = False
for name, param in model.named_parameters():
    if torch.isnan(param).any():
        print(f"Warning: {name} contains NaN")
        has_nan = True

if not has_nan:
    print("Model weights initialized cleanly. Ready to train.")
else:
    print("Initialization failed due to NaNs.")

Model weights initialized cleanly. Ready to train.


In [8]:
# Start variables
model.train()
train_accuracies = []
validation_accuracies = []
train_losses = []

# Hyperparameters
num_epochs = 5
batch_size = 32 
context_window = 3

loss_fn = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

validation_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False, drop_last=False)

print(f"Starting training for {num_epochs} epochs...")

step = 0

# Start training loop
for epoch in range(num_epochs):
    print(f"\n--- Starting Epoch {epoch+1} of {num_epochs} ---")
    
    # loader for every epoch
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    
    epoch_train_losses = []
    epoch_train_preds = []
    epoch_train_targets = []
    
    model.train()
    for x, _, _, _ in train_loader:
        optimizer.zero_grad()
        batch_loss = 0.0
        batch_preds = []
        batch_targets = []
        valid_steps = 0
        
        seq_len = x.shape[1]
        
        # slide across the sequence length
        for t in range(seq_len - context_window):
            inputs = x[:, t : t + context_window]               
            targets = x[:, t + context_window]                  
            
            if (targets != IGNORE_INDEX).sum() == 0:
                continue
                
            output = model(inputs)                              
            loss = loss_fn(output, targets)
            
            # check for NaN or Inf
            if not torch.isnan(loss) and not torch.isinf(loss):
                batch_loss = batch_loss + loss
                valid_steps += 1
            
            predictions = output.max(1)[1]
            batch_preds.append(predictions.detach())
            batch_targets.append(targets.detach())
        
        if valid_steps > 0:
            batch_loss = batch_loss / valid_steps
            
            # check batch loss validity
            if not torch.isnan(batch_loss) and not torch.isinf(batch_loss):
                batch_loss.backward()
                
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                
                optimizer.step()
                step += 1
                epoch_train_losses.append(batch_loss.item())
            else:
                optimizer.zero_grad()
                continue
        else:
            continue
        
        if batch_preds:
            epoch_train_preds.append(torch.cat(batch_preds))
            epoch_train_targets.append(torch.cat(batch_targets))
    
    # metrics
    train_loss_avg = np.mean(epoch_train_losses) if epoch_train_losses else 0.0
    train_losses.append(train_loss_avg)
    
    if epoch_train_preds:
        all_train_preds = torch.cat(epoch_train_preds)
        all_train_targets = torch.cat(epoch_train_targets)
        fl = all_train_targets != IGNORE_INDEX
        train_acc = metrics.accuracy_score(all_train_targets[fl].cpu(), all_train_preds[fl].cpu()) if fl.sum() > 0 else 0.0
    else:
        train_acc = 0.0
    train_accuracies.append(train_acc)
    
    # compute validation accuracy
    right_pred, total_pred = 0, 0
    with torch.no_grad():
        model.eval()
        for x_val, _, _, _ in validation_loader:
            seq_len_val = x_val.shape[1]
            for t in range(seq_len_val - context_window):
                inputs_val = x_val[:, t : t + context_window]
                targets_val = x_val[:, t + context_window]
                
                if (targets_val != IGNORE_INDEX).sum() == 0:
                    continue
                    
                output_val = model(inputs_val)
                predictions_val = output_val.max(1)[1]
                
                fl_val = targets_val != IGNORE_INDEX
                right_pred += (predictions_val[fl_val] == targets_val[fl_val]).sum().item()
                total_pred += fl_val.sum().item()
        model.train()
        
    val_acc = (right_pred / total_pred) if total_pred > 0 else 0.0
    validation_accuracies.append(val_acc)
    
    print(f"Finished Epoch {epoch+1} | Total Steps: {step}")
    print(f"   Training Loss:       {train_loss_avg:.4f}")
    print(f"   Training Accuracy:   {train_acc:.4f}")
    print(f"   Validation Accuracy: {val_acc:.4f}\n")

print("Finished training across all epochs successfully.")

Starting training for 5 epochs...

--- Starting Epoch 1 of 5 ---
Finished Epoch 1 | Total Steps: 501
   Training Loss:       1.6211
   Training Accuracy:   0.3906
   Validation Accuracy: 0.4129


--- Starting Epoch 2 of 5 ---
Finished Epoch 2 | Total Steps: 1002
   Training Loss:       1.4376
   Training Accuracy:   0.4253
   Validation Accuracy: 0.4344


--- Starting Epoch 3 of 5 ---
Finished Epoch 3 | Total Steps: 1503
   Training Loss:       1.4130
   Training Accuracy:   0.4366
   Validation Accuracy: 0.4221


--- Starting Epoch 4 of 5 ---
Finished Epoch 4 | Total Steps: 2004
   Training Loss:       1.4063
   Training Accuracy:   0.4399
   Validation Accuracy: 0.4371


--- Starting Epoch 5 of 5 ---
Finished Epoch 5 | Total Steps: 2505
   Training Loss:       1.3911
   Training Accuracy:   0.4446
   Validation Accuracy: 0.4407

Finished training across all epochs successfully.
